[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/03_linear.ipynb)

# 🟡 中等难度：简单线性层

实现一个全连接线性层：**y = xW^T + b**

### 函数签名
```python
class SimpleLinear:
    def __init__(self, in_features: int, out_features: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### 要求
- `self.weight`：形状 `(out_features, in_features)`，初始化为 `randn * (1/√in_features)`
- `self.bias`：形状 `(out_features,)`，初始化为全零
- 两者都必须设置 `requires_grad=True`
- `forward(x)` 需要计算 `x @ W^T + b`
- **禁止**使用 `torch.nn.Linear`

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

/Users/xiaohui/Documents/project/TorchCode/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


- $1/\sqrt{in\_features}$ 初始化实际上是 Xavier/Glorot初始化的一种变体，它能确保前向传播时信号的方差不会过快地爆炸或消失。如果初始化权重过大，信号会逐层放大导致梯度爆炸；如果权重过小，信号会逐层衰减导致梯度消失。Xavier 初始化通过数学推导，找到了一个“平衡点”。核心目标是使神经网络中每一层输出的方差等于其输入的方差。
- 代码中使用了 requires_grad_(True)。在 PyTorch 中，带有下划线的方法（如 .requires_grad_() 或 .zero_()）是原地操作，直接修改 Tensor 属性而不需要重新赋值。
- x @ self.weight.t() 自动处理了 Batch 维度的矩阵乘法。即使 x 是多维的（比如图像序列），只要最后一维匹配 in_features，它就能正常工作。

1. 初始化权重：形状 (out_features, in_features),遵循 randn * (1 / sqrt(in_features))
2. 初始化偏置：形状 (out_features,)，全零
3. 设置 requires_grad=True 以支持反向传播

| 初始化方法       | 方差 (σ²)          | 适用场景                              |
|------------------|---------------------|---------------------------------------|
| Lecun           | 1/n_in             | 早期网络，考虑前向传播方差一致         |
| Xavier (Glorot) | 2/(n_in + n_out)   | 现代全连接层，兼顾前向和反向传播       |
| He (Kaiming)    | 2/n_in             | ReLU 及其变体（目前最主流）            |

In [ ]:
# ✏️ 在这里实现你的代码

class SimpleLinear:
    def __init__(self, in_features: int, out_features: int):
        '''
        weight = [out_features, in_features], W 的实际形状
        '''
        # 1. 初始化权重：形状 (out_features, in_features)
        # 遵循 randn * (1 / sqrt(in_features))
        stdv = 1.0 / math.sqrt(in_features)
        self.weight = torch.randn(out_features, in_features) * stdv
        
        # 2. 初始化偏置：形状 (out_features,)，全零
        self.bias = torch.zeros(out_features)
        
        # 3. 设置 requires_grad=True 以支持反向传播
        self.weight.requires_grad_(True)
        self.bias.requires_grad_(True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        计算算式: y = x @ W^T + b
        x 形状: (batch_size, in_features)
        W^T 形状: (in_features, out_features)

        [batch_size, in_features] @ [in_features, out_features] -> [batch_size, out_features] 相当于 batch_size 个 [1, out_features] 的输出
        """
        # 使用 @ 运算符进行矩阵乘法，.t() 或 .T 进行转置
        return x @ self.weight.t() + self.bias

# --- 快速验证 ---
# 定义输入 (batch_size=3, in_features=4)
x = torch.randn(3, 4)
layer = SimpleLinear(4, 2)

# 执行前向传播
output = layer.forward(x)

print(f"权重形状: {layer.weight.shape}") # 应为 (2, 4)
print(f"偏置形状: {layer.bias.shape}")   # 应为 (2,)
print(f"输出形状: {output.shape}")       # 应为 (3, 2)
print(f"梯度检查: {layer.weight.requires_grad}") # 应为 True

In [5]:
x = torch.tensor([[1,2], [6,8]]) # (2,2)

bias = torch.tensor([10, 20]) # (2)
bias_ = torch.tensor([[10,20], [10, 20]])

out = x + bias
out_ = x + bias_
print(out) # bias 的广播方向是增加维度面上重复
print(out_)

tensor([[11, 22],
        [16, 28]])
tensor([[11, 22],
        [16, 28]])


如果你这个类能像标准的 nn.Module 那样通过 model.parameters() 自动管理参数，我们需要将其继承自 torch.nn.Module 并将 Tensor 转换为 nn.Parameter。当你把一个 Tensor 赋值给 nn.Parameter 时，PyTorch 会自动将其标记为 requires_grad=True 并在 model.parameters() 中收集它。如果你只用普通的 Tensor，优化器将无法“看到”它。

In [ ]:
from torch import nn

class SimpleLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # 使用 nn.Parameter 包装 Tensor
        # 这样该参数会自动注册到 self.parameters() 中
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))

        # 执行你要求的重置参数
        self.reset_parameters()

    def reset_parameters(self):
        # 权重初始化：randn * (1 / sqrt(in_features))
        stdv = 1.0 / math.sqrt(self.in_features)
        nn.init.normal_(self.weight, mean=0, std=stdv)
        
        # 偏置初始化：全零
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算 x @ W^T + b
        # 即使输入 x 是 (batch, ..., in_features)，矩阵乘法也能自动处理
        return x @ self.weight.t() + self.bias

model = nn.Sequential(
    SimpleLinear(10, 20),
    nn.ReLU(),
    SimpleLinear(20, 5)
)

# 1. 自动管理参数：优化器可以直接获取参数
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 2. 设备一键切换：
# model.to("cuda") # 内部的 weight 和 bias 会自动转到 GPU

# 3. 状态检查
print(f"参数列表: {[name for name, _ in model.named_parameters()]}")

In [ ]:
# 🧪 调试
layer = SimpleLinear(8, 4)
print("W 的形状：", layer.weight.shape)  # 应该是 (4, 8)
print("b 的形状：", layer.bias.shape)    # 应该是 (4,)

x = torch.randn(2, 8)
y = layer.forward(x)
print("输出形状：", y.shape)             # 应该是 (2, 4)

In [ ]:
# ✅ 提交验证
from torch_judge import check
check("linear")